# FIFA World Cup 2026 — Poisson Match Predictor

## About

**Purpose:** Predict Win / Draw / Loss probabilities for international football matches using a Poisson goals model.<br>
**Author:** Ganapathy K<br>
**Date:** 2026-06-04<br>
**Notes:** Data source = martj42 international results (1872–2026, ~49,378 matches). Strengths are recency-weighted so current form outweighs historical reputation.<br>
**Description:** This notebook is the data-ingestion stage. It loads the raw international match-results CSV, inspects it, and prepares a clean played-matches table that later notebooks use to compute team attack/defence strengths and Poisson scoreline probabilities.

### Change Control

| Date       | Version | Author      | Changes         |
|------------|---------|-------------|-----------------|
| 2026-06-04 | 1.0     | Ganapathy K | Initial version |


## 1. Setup

In [1]:
%load_ext autoreload
%autoreload 2

### 1.1 Imports

In [2]:
import pandas as pd
from pathlib import Path

### 1.2 Config

In [3]:
INTERNATIONAL_RESULTS_PATH = Path(r"D:/Data Science/Datasets/Football/international_results.csv")

PROCESSED_DATA_DIR = Path(r"D:/Data Science/Visual Studio Code/fifa_wc_2026_poisson/data/processed")
PLAYED_MATCHES_PATH = PROCESSED_DATA_DIR / "played_matches.parquet"

## 2. Data Ingestion
### 2.1 Load match history

In [4]:
match_history = pd.read_csv(INTERNATIONAL_RESULTS_PATH, parse_dates=["date"])

### 2.2 Inspect

In [5]:
match_history.info()
match_history.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49378 entries, 0 to 49377
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   date        49378 non-null  datetime64[ns]
 1   home_team   49378 non-null  object        
 2   away_team   49378 non-null  object        
 3   home_score  49306 non-null  float64       
 4   away_score  49306 non-null  float64       
 5   tournament  49378 non-null  object        
 6   city        49378 non-null  object        
 7   country     49378 non-null  object        
 8   neutral     49378 non-null  bool          
dtypes: bool(1), datetime64[ns](1), float64(2), object(5)
memory usage: 3.1+ MB


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


## 3. Data Preparation

Future fixtures (scheduled but not yet played) have blank scores. Drop any row missing a score, then cast scores back to integers.

In [6]:
played_matches = match_history.dropna(subset=["home_score", "away_score"]).copy()
played_matches["home_score"] = played_matches["home_score"].astype(int)
played_matches["away_score"] = played_matches["away_score"].astype(int)

print(f"Total rows loaded:   {len(match_history)}")
print(f"Played matches kept: {len(played_matches)}")
print(f"Unplayed dropped:    {len(match_history) - len(played_matches)}")

Total rows loaded:   49378
Played matches kept: 49306
Unplayed dropped:    72


## 4. Save

Write the clean played-matches table to `data/processed/` as parquet for notebook 02 (team strengths).

In [7]:
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
played_matches.to_parquet(PLAYED_MATCHES_PATH, index=False)
print(f"Saved {len(played_matches)} played matches \u2192 {PLAYED_MATCHES_PATH}")

Saved 49306 played matches → D:\Data Science\Visual Studio Code\fifa_wc_2026_poisson\data\processed\played_matches.parquet
